In [10]:
from sklearn.mixture import GaussianMixture
import numpy as np

W_FEAT = np.array([1.0, 0.7, 0.3], dtype=float)   
W_MEAN = np.array([0.6, 0.3, 0.1], dtype=float)   

"""
X: (N,3) = [p_face, p_audio, p_text] in [0,1]
"""
def fit_gmm(X, random_state=42):
    gmm = GaussianMixture(n_components=2, covariance_type="diag", random_state=random_state)
    Xs = X * np.sqrt(W_FEAT)   # 축별 스케일만 곱해서 영향도 조절
    gmm.fit(Xs)
    return gmm

"""
군집 0/1 중 '우울'을 고름: 가중평균 점수가 큰 군집 = 우울
"""
def pick_depressed_cluster(gmm, X):
    Xs = X * np.sqrt(W_FEAT)
    labels = gmm.predict(Xs)
    scores = []
    for k in range(gmm.n_components):
        idx = labels == k
        if not np.any(idx):
            scores.append(-np.inf)
        else:
            scores.append((X[idx] @ W_MEAN).mean())
    return int(np.argmax(scores))

"""
각 샘플이 '우울 군집'일 확률 (posterior)
"""
def predict_dep_prob(gmm, X, dep_cluster):
    Xs = X * np.sqrt(W_FEAT)
    P = gmm.predict_proba(Xs)              
    return P[:, dep_cluster]               

def simple_label(p, high=0.80, low=0.20):
    if p >= high: return "depressed"
    if p <= low:  return "nondepressed"
    return "uncertain"


def predict_one(x, gmm, dep_idx, high=0.80, low=0.20):
    """
    x: (3,) = [p_face, p_audio, p_text]
    반환: (prob, label)  e.g., (0.87, 'depressed')
    """
    p = predict_dep_prob(gmm, np.array([x], float), dep_idx)[0]
    label = simple_label(np.array([p]), high=high, low=low)
    return float(p), str(label)

def predict_batch(X_new, gmm, dep_idx, high=0.80, low=0.20):
    """
    X_new: (N,3)
    반환: (probs (N,), labels (N,))
    """
    probs = predict_dep_prob(gmm, np.array(X_new, float), dep_idx)
    labels = simple_label(probs, high=high, low=low)
    return probs, labels

In [11]:
X = np.array([
    [0.85, 0.30, 0.78],
    [0.82, 0.35, 0.80],
    [0.10, 0.20, 0.15],
    [0.12, 0.18, 0.22],
    [0.70, 0.45, 0.72],
    [0.20, 0.10, 0.25],
], float)

gmm = fit_gmm(X)
dep_idx = pick_depressed_cluster(gmm, X)     # 어떤 군집이 '우울'인지 결정
X_new = np.array([
    [0.78, 0.40, 0.76],
    [0.18, 0.12, 0.20],
    [0.55, 0.45, 0.50],
], float)

p_dep = predict_dep_prob(gmm, X_new, dep_idx)  # 우울 확률
labels = [simple_label(p) for p in p_dep]
print(p_dep)   
print(labels)  

[1.00000000e+00 2.16098999e-83 1.00000000e+00]
['depressed', 'nondepressed', 'depressed']


In [13]:
p, lbl = predict_one([0.78, 0.40, 0.76], gmm, dep_idx)  # 단일 샘플
print(p, lbl)  # 예: 0.87 'depressed'


1.0 depressed
